<a href="https://colab.research.google.com/github/MusaR10/AAI2025/blob/2026fall/Excercise_3_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3: Self-Reflection Prompt for Improving Output

**Goal:** Ask the AI to critique and improve its own summary against explicit requirements (critique-and-revise).

**Flow:**
1. **First draft**: Gemini summarizes an article using a simple prompt
2. **Self-critique**: Gemini checks its own summary against five explicit criteria: accuracy, clarity, audience, length, and formatting
3. **Revise**: Gemini rewrites the summary to fix every problem it found
4. **Re-check**: the revised summary is critiqued again; if it still falls short, it is revised once more (maximum 2 revisions)
5. **Compare**: the before and after summaries are shown side by side with scores and automatic checks

**Tools:** Google Colab, Python 3, Google Gemini API (`gemini-3.1-flash-lite`), pandas

In [2]:
import google.generativeai as genai
from google.colab import userdata, files
from IPython.display import display, Markdown
from PIL import Image as PILImage
import time
import os
import json
import re
import pandas as pd

# Connect to Gemini
genai.configure(api_key=userdata.get("Prompt_Engineering_Key"))

model = genai.GenerativeModel("gemini-3.1-flash-lite")

print("Gemini initialized successfully.")

Gemini initialized successfully.


## Helper functions
`ask()` sends a prompt to Gemini and can return plain text or parsed JSON. `show()` prints each step's output with a label.

In [3]:
def ask(prompt, temperature=0.3, as_json=False, retries=3):
    """Send one prompt to Gemini and return the text (or parsed JSON)."""
    config = {"temperature": temperature}
    if as_json:
        config["response_mime_type"] = "application/json"

    for attempt in range(retries):
        try:
            response = model.generate_content(prompt, generation_config=config)
            time.sleep(3)  # small pause to stay under free-tier rate limits
            text = response.text.strip()
            if as_json:
                text = text.replace("```json", "").replace("```", "").strip()
                return json.loads(text)
            return text
        except Exception as e:
            print(f"  (attempt {attempt + 1} failed: {str(e)[:100]} ... retrying)")
            time.sleep(20)
    raise RuntimeError("Gemini call failed after several retries.")


def show(label, data):
    """Print one step's output with a label."""
    print(f"\n--- {label} ---")
    print(json.dumps(data, indent=2) if isinstance(data, (dict, list)) else data)

## Source article
The text Gemini will summarize: a short local news article about a city composting program. It contains many specific numbers, which makes accuracy easy to check.

In [7]:
ARTICLE = """Maple Grove's Curbside Composting Pilot Cuts Trash by Nearly a Third

In March 2025, the city of Maple Grove launched a 12-month curbside composting pilot in 3 neighborhoods covering about 4,200 households. Each household received a free 13-gallon green bin and a small kitchen container, and food scraps were collected every week on the same day as regular trash pickup. Participation was voluntary.

By the time the pilot ended in February 2026, 2,730 households, about 65%, were setting out their green bins at least once a month. Together they kept an estimated 1,150 tons of food waste out of the county landfill. City data showed that garbage from the three pilot neighborhoods fell 31% compared with the same months a year earlier, while garbage in the rest of the city fell only 2%.

The pilot cost $486,000, which covered the bins, a leased collection truck, and two part-time drivers. That works out to roughly $116 per household for the year. Officials estimate that about $140,000 of that cost was offset by lower landfill fees.

The program had early problems. In the first 2 months, 18% of green bins contained items that cannot be composted, mostly plastic bags. After the city began leaving tags on bins explaining what went wrong, that rate fell to 6% by the sixth month. The finished compost was given back to residents at 4 free pickup events, which about 900 people attended.

In a survey of 1,100 participants, 82% said they were satisfied with the program. However, 11% complained about odors during the summer, and apartment buildings were largely left out because they use shared dumpsters. Council members say the apartment problem must be solved before the program grows.

The city council will vote on October 14, 2026, on whether to expand the program to all 21,000 households in Maple Grove, at a projected cost of $2.1 million per year. Supporters point to the drop in landfill waste. Critics, including council member Dana Ruiz, question whether the city can afford the expansion while it faces a $3.4 million budget gap."""

print(f"Article loaded: {len(ARTICLE.split())} words.")

Article loaded: 344 words.


## Requirements (the critique criteria)
The five explicit criteria the summary is judged against. They are inserted into both the critique prompt and the revision prompt.

In [4]:
REQUIREMENTS = """1. ACCURACY: Every fact must match the article. Use numbers exactly as written in the article (do not round or convert them). Do not add any information, opinions, or predictions that are not in the article.

2. CLARITY: Use plain language that a 9th grader can follow. Replace or explain technical words such as "diverted" or "contamination". Keep sentences short.

3. AUDIENCE: Write for Maple Grove residents who know nothing about the program and are deciding whether to attend the council vote. The summary MUST include: the pilot's main result, its cost, at least one concern, and the date of the vote. Keep a neutral tone that presents both a benefit and a concern.

4. LENGTH: 70 to 100 words in total.

5. FORMATTING:
   - Line 1: a bold headline of 12 words or fewer, written as **Headline**
   - Then exactly 3 bullet points, each starting with "- "
   - Last line: one sentence starting with "Bottom line:"
"""

print(REQUIREMENTS)

1. ACCURACY: Every fact must match the article. Use numbers exactly as written in the article (do not round or convert them). Do not add any information, opinions, or predictions that are not in the article.

2. CLARITY: Use plain language that a 9th grader can follow. Replace or explain technical words such as "diverted" or "contamination". Keep sentences short.

3. AUDIENCE: Write for Maple Grove residents who know nothing about the program and are deciding whether to attend the council vote. The summary MUST include: the pilot's main result, its cost, at least one concern, and the date of the vote. Keep a neutral tone that presents both a benefit and a concern.

4. LENGTH: 70 to 100 words in total.

5. FORMATTING:
   - Line 1: a bold headline of 12 words or fewer, written as **Headline**
   - Then exactly 3 bullet points, each starting with "- "
   - Last line: one sentence starting with "Bottom line:"



## Step 1: First draft
A simple, generic summary prompt, the kind most people write first. This gives the "before" version that self-reflection will improve.

In [8]:
def first_draft(article):
    prompt = f"""Summarize the following article.

{article}"""
    return ask(prompt, temperature=0.3)


draft = first_draft(ARTICLE)
show("STEP 1 - FIRST DRAFT SUMMARY", draft)


--- STEP 1 - FIRST DRAFT SUMMARY ---
Maple Grove’s 12-month curbside composting pilot program successfully diverted 1,150 tons of food waste from landfills, resulting in a 31% reduction in trash for participating neighborhoods. While the program saw high resident satisfaction (82%) and improved sorting accuracy after initial contamination issues, it faced challenges regarding summer odors and the exclusion of apartment buildings. 

With the pilot costing $486,000—partially offset by reduced landfill fees—the city council is now weighing a $2.1 million annual expansion to all households. A final vote is scheduled for October 14, 2026, though the decision remains contentious due to the city's current $3.4 million budget deficit.


## Automatic checks
Language models often miscount words and miss small formatting errors, so length, format, and required content are also verified with code. Every number in the summary is also checked against the numbers in the article.

In [9]:
def numbers_in(text):
    """Find numbers like 4,200 / $486,000 / 65% / 2.1 and normalize them."""
    found = re.findall(r"\$?\d[\d,]*(?:\.\d+)?%?", text)
    return {n.replace("$", "").replace(",", "").rstrip(".") for n in found}


ARTICLE_NUMBERS = numbers_in(ARTICLE)


def auto_check(summary):
    lines = [line.strip() for line in summary.strip().splitlines() if line.strip()]
    plain = summary.replace("**", "").replace("- ", " ")
    words = len(plain.split())
    unknown_numbers = sorted(numbers_in(summary) - ARTICLE_NUMBERS)

    return {
        "Word count": words,
        "Length 70-100": 70 <= words <= 100,
        "Bold headline": bool(lines) and lines[0].startswith("**") and lines[0].endswith("**")
                         and len(lines[0].replace("**", "").split()) <= 12,
        "Exactly 3 bullets": sum(1 for line in lines if line.startswith("- ")) == 3,
        "Bottom line last": bool(lines) and lines[-1].startswith("Bottom line:"),
        "Has cost": "$" in summary,
        "Has vote date": bool(re.search(r"Oct(ober|\.)?\s*14", summary)),
        "Numbers match article": not unknown_numbers,
    }


show("AUTOMATIC CHECKS - FIRST DRAFT", auto_check(draft))


--- AUTOMATIC CHECKS - FIRST DRAFT ---
{
  "Word count": 100,
  "Length 70-100": true,
  "Bold headline": false,
  "Exactly 3 bullets": false,
  "Bottom line last": false,
  "Has cost": true,
  "Has vote date": true,
  "Numbers match article": true
}


## Step 2: Self-critique prompt
Gemini reviews its own summary against the five criteria. It must check every factual claim against the article, score each criterion from 1 to 5, quote the exact words that cause each problem, and say how to fix it.

In [10]:
def self_critique(article, summary):
    prompt = f"""ROLE: You are a strict editor reviewing a summary that you wrote earlier. Critique it honestly against the requirements. Do not defend it.

ORIGINAL ARTICLE:
\"\"\"{article}\"\"\"

SUMMARY TO REVIEW:
\"\"\"{summary}\"\"\"

REQUIREMENTS:
{REQUIREMENTS}

TASK:
1. Accuracy check: list each factual claim in the summary and check it against the article.
2. Score each of the five criteria from 1 (fails) to 5 (fully meets).
3. List every specific problem and exactly how to fix it.

Return JSON with exactly these keys:
- "claim_check": a list of objects with keys "claim", "supported" (true or false), and "note"
- "scores": an object with integer keys "accuracy", "clarity", "audience", "length", "formatting"
- "word_count": your count of the words in the summary
- "problems": a list of strings, each written as "Problem: ... | Fix: ..."
- "meets_all_requirements": true only if every score is 5

RULES:
- Judge only against the article and the requirements, not general knowledge.
- Quote the exact words from the summary that cause each problem.
- If there are no problems, return an empty "problems" list."""
    return ask(prompt, temperature=0, as_json=True)

## Step 3: Revision prompt
Gemini rewrites its summary using its own critique. It must fix every listed problem, keep what was already correct, and return only the revised summary in the required format.

In [11]:
def revise(article, summary, critique):
    prompt = f"""ROLE: You are revising your own summary based on an editor's critique.

ORIGINAL ARTICLE:
\"\"\"{article}\"\"\"

YOUR PREVIOUS SUMMARY:
\"\"\"{summary}\"\"\"

CRITIQUE OF THAT SUMMARY:
{json.dumps(critique, indent=2)}

REQUIREMENTS:
{REQUIREMENTS}

TASK: Write a revised summary that fixes every problem in the critique and meets all five requirements.

RULES:
- Correct or remove every claim marked "supported": false. Use only facts from the article.
- Keep anything the critique did not flag as a problem.
- Count your words before answering: the total must be 70 to 100 words.
- Return ONLY the revised summary in the required format, with no notes or explanations."""
    return ask(prompt, temperature=0.3)

## The self-reflection loop
Connects the steps: critique the current summary → revise it → critique the new version. The loop stops when the critique finds no problems and the automatic checks all pass, or after 2 revisions.

In [12]:
def reflection_loop(article, draft, max_revisions=2):
    versions = [{"Version": "v1 (first draft)", "summary": draft}]

    for round_number in range(1, max_revisions + 2):
        current = versions[-1]
        print("\n" + "=" * 75)
        print(f"REVIEWING {current['Version'].upper()}")
        print("=" * 75)

        critique = self_critique(article, current["summary"])
        checks = auto_check(current["summary"])
        current["critique"] = critique
        current["checks"] = checks

        show("SELF-CRITIQUE: SCORES", critique.get("scores"))
        show("SELF-CRITIQUE: ACCURACY CHECK", critique.get("claim_check"))
        show("SELF-CRITIQUE: PROBLEMS FOUND", critique.get("problems"))
        show("AUTOMATIC CHECKS", checks)

        checks_pass = all(v for k, v in checks.items() if k != "Word count")
        if critique.get("meets_all_requirements") and checks_pass:
            print(f"\n✅ {current['Version']} meets every requirement. Stopping.")
            break
        if len(versions) > max_revisions:
            print(f"\n⚠️ Reached the limit of {max_revisions} revisions.")
            break

        revised = revise(article, current["summary"], critique)
        label = f"v{len(versions) + 1} (revision {len(versions)})"
        versions.append({"Version": label, "summary": revised})
        show(f"REVISED SUMMARY: {label}", revised)

    return versions

In [13]:
versions = reflection_loop(ARTICLE, draft)
display(Markdown("### BEFORE: first draft"))
display(Markdown(versions[0]["summary"]))
display(Markdown("---"))
display(Markdown(f"### AFTER: {versions[-1]['Version']}"))
display(Markdown(versions[-1]["summary"]))

rows = []
for v in versions:
    scores = v["critique"].get("scores", {})
    checks = v["checks"]
    rows.append({
        "Version": v["Version"],
        **{k.capitalize(): scores.get(k) for k in ["accuracy", "clarity", "audience", "length", "formatting"]},
        "AI word count": v["critique"].get("word_count"),
        **checks,
    })

comparison = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
display(comparison)


REVIEWING V1 (FIRST DRAFT)

--- SELF-CRITIQUE: SCORES ---
{
  "accuracy": 5,
  "clarity": 3,
  "audience": 4,
  "length": 5,
  "formatting": 2
}

--- SELF-CRITIQUE: ACCURACY CHECK ---
[
  {
    "claim": "12-month curbside composting pilot",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "diverted 1,150 tons of food waste",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "31% reduction in trash",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "82% satisfaction",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "pilot costing $486,000",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "$2.1 million annual expansion",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "vote on October 14, 2026",
    "supported": true,
    "note": "Matches article."
  },
  {
    "claim": "$3.4 million budget deficit",
    "supported": true,
  

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 756.26ms



--- SELF-CRITIQUE: SCORES ---
{
  "accuracy": 5,
  "clarity": 5,
  "audience": 5,
  "length": 4,
  "formatting": 5
}

--- SELF-CRITIQUE: ACCURACY CHECK ---
[
  {
    "claim": "Maple Grove\u2019s 12-month pilot kept 1,150 tons of food waste out of landfills",
    "supported": true,
    "note": "Matches article data."
  },
  {
    "claim": "cutting trash in participating neighborhoods by 31%",
    "supported": true,
    "note": "Matches article data."
  },
  {
    "claim": "The pilot cost $486,000",
    "supported": true,
    "note": "Matches article data."
  },
  {
    "claim": "residents reported issues like summer odors and difficulty including apartment buildings",
    "supported": true,
    "note": "Matches article data."
  },
  {
    "claim": "Sorting accuracy improved after the city addressed items that cannot be composted",
    "supported": true,
    "note": "Matches article data."
  },
  {
    "claim": "82% of participants expressed satisfaction with the program",
    "supporte

### BEFORE: first draft

Maple Grove’s 12-month curbside composting pilot program successfully diverted 1,150 tons of food waste from landfills, resulting in a 31% reduction in trash for participating neighborhoods. While the program saw high resident satisfaction (82%) and improved sorting accuracy after initial contamination issues, it faced challenges regarding summer odors and the exclusion of apartment buildings. 

With the pilot costing $486,000—partially offset by reduced landfill fees—the city council is now weighing a $2.1 million annual expansion to all households. A final vote is scheduled for October 14, 2026, though the decision remains contentious due to the city's current $3.4 million budget deficit.

---

### AFTER: v3 (revision 2)

**Maple Grove Considers Expanding Curbside Composting Program**

- The 12-month pilot kept 1,150 tons of food waste out of landfills, reducing trash in participating neighborhoods by 31%.
- The program cost $486,000, though residents reported summer odors and noted that apartment buildings were largely excluded.
- Sorting accuracy improved after the city provided feedback on non-compostable items, and 82% of participants expressed satisfaction with the service.

Bottom line: The city council will vote on October 14, 2026, on a $2.1 million expansion despite the city’s $3.4 million budget gap.

,Version,Accuracy,Clarity,Audience,Length,Formatting,AI word count,Word count,Length 70-100,Bold headline,Exactly 3 bullets,Bottom line last,Has cost,Has vote date,Numbers match article
0,v1 (first draft),5,3,4,5,2,100,100,True,False,False,False,True,True,True
1,v2 (revision 1),5,5,5,4,5,103,87,True,True,True,True,True,True,True
2,v3 (revision 2),5,5,5,4,5,103,86,True,True,True,True,True,True,True
